# Prompt Engineering Lab
## TechFlow Solutions - Customer Service Chatbot Improvement

**Tasks covered:**
1. Sentiment Analysis
2. Product Description Generation
3. Data Extraction

---
## Part 1: Environment Setup

In [1]:
# Install dependencies if needed
# !pip install openai

import os
import json
import time
from collections import Counter
from openai import OpenAI

# Initialize client — set your key via environment variable: OPENAI_API_KEY
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Helper: single call ──────────────────────────────────────────────────────
def call_openai(prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.7) -> str:
    """Call OpenAI and return the text response."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()


# ── Helper: run N times ──────────────────────────────────────────────────────
def run_prompt_n_times(prompt: str, n: int = 5, delay: float = 0.3) -> list[str]:
    """Run the same prompt n times and return all responses."""
    results = []
    for i in range(n):
        result = call_openai(prompt)
        results.append(result)
        time.sleep(delay)  # be polite to the API
    return results


# ── Helper: consistency analysis ─────────────────────────────────────────────
def analyze_consistency(results: list[str], label: str = "") -> None:
    """Print basic consistency stats for a list of responses."""
    normalized = [r.strip().lower() for r in results]
    counts = Counter(normalized)
    most_common, top_count = counts.most_common(1)[0]
    consistency_pct = (top_count / len(results)) * 100

    print(f"\n{'='*60}")
    print(f"Consistency Analysis{' — ' + label if label else ''}")
    print(f"{'='*60}")
    print(f"  Total runs      : {len(results)}")
    print(f"  Unique responses: {len(counts)}")
    print(f"  Top response    : {most_common!r}")
    print(f"  Consistency     : {consistency_pct:.1f}%")
    print("\n  All responses:")
    for i, r in enumerate(results, 1):
        print(f"    [{i:02d}] {r[:120]}")


print("✅ Environment ready.")

✅ Environment ready.


---
## Part 1 — Step 2: Initial (v1) Zero-Shot Prompts

In [2]:
# ── Task 1: Sentiment Analysis — v1 ─────────────────────────────────────────
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis v1 — single run:")
print(result)

Sentiment Analysis v1 — single run:
This customer message can be classified as **Positive Feedback**.


In [3]:
# ── Task 2: Product Description — v1 ────────────────────────────────────────
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

result = call_openai(product_prompt_v1)
print("Product Description v1 — single run:")
print(result)

Product Description v1 — single run:
**Product Description: Wireless Ergonomic Mouse - $29.99**

Elevate your productivity and comfort with our Wireless Ergonomic Mouse, designed to keep you moving seamlessly through your day. Priced at just $29.99, this sleek and stylish mouse combines advanced technology with user-friendly features to enhance your computer experience.

**Key Features:**

- **Ergonomic Design**: Crafted for long hours of use, our mouse fits perfectly in your hand, reducing strain and providing maximum comfort whether you're working, gaming, or browsing.

- **Wireless Convenience**: Say goodbye to tangled cords! With a reliable 2.4 GHz wireless connection, enjoy a clutter-free workspace and the freedom to move without restrictions. A USB receiver is included for easy plug-and-play setup.

- **High Precision Tracking**: Equipped with a high-precision optical sensor, this mouse delivers smooth and accurate tracking on various surfaces, ensuring every click counts. Choose

In [5]:
# ── Task 3: Data Extraction — v1 ────────────────────────────────────────────
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

result = call_openai(extraction_prompt_v1)
print("Data Extraction v1 — single run:")
print(result)

Data Extraction v1 — single run:
Here are the key details extracted from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged


---
## Part 2 — Steps 3-5: Systematic Testing (5 / 10 / 15 runs)

In [6]:
# ── 5-run baseline ───────────────────────────────────────────────────────────
print("Running each v1 prompt 5 times ...")

sentiment_5  = run_prompt_n_times(sentiment_prompt_v1,  n=5)
product_5    = run_prompt_n_times(product_prompt_v1,    n=5)
extraction_5 = run_prompt_n_times(extraction_prompt_v1, n=5)

analyze_consistency(sentiment_5,  "Sentiment v1 — 5 runs")
analyze_consistency(product_5,    "Product v1   — 5 runs")
analyze_consistency(extraction_5, "Extraction v1 — 5 runs")

Running each v1 prompt 5 times ...

Consistency Analysis — Sentiment v1 — 5 runs
  Total runs      : 5
  Unique responses: 3
  Top response    : 'the customer message can be classified as "positive feedback" or "customer satisfaction."'
  Consistency     : 60.0%

  All responses:
    [01] The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
    [02] The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
    [03] The customer message can be classified as "Positive Feedback" or "Satisfaction."
    [04] The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
    [05] This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Consistency Analysis — Product v1   — 5 runs
  Total runs      : 5
  Unique responses: 5
  Top response    : '**product description: wireless precision mouse**\n\nelevate your productivity with our sleek and stylish wireless pr

In [7]:
# ── 10-run test ──────────────────────────────────────────────────────────────
print("Running each v1 prompt 10 times ...")

sentiment_10  = run_prompt_n_times(sentiment_prompt_v1,  n=10)
product_10    = run_prompt_n_times(product_prompt_v1,    n=10)
extraction_10 = run_prompt_n_times(extraction_prompt_v1, n=10)

analyze_consistency(sentiment_10,  "Sentiment v1 — 10 runs")
analyze_consistency(product_10,    "Product v1   — 10 runs")
analyze_consistency(extraction_10, "Extraction v1 — 10 runs")

Running each v1 prompt 10 times ...

Consistency Analysis — Sentiment v1 — 10 runs
  Total runs      : 10
  Unique responses: 8
  Top response    : 'the customer message can be classified as **positive feedback** or **customer satisfaction**.'
  Consistency     : 20.0%

  All responses:
    [01] The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [02] This customer message can be classified as positive feedback or a positive review.
    [03] This customer message can be classified as **positive feedback** or **customer satisfaction**.
    [04] The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [05] The customer message can be classified as "Positive Feedback" or "Satisfaction."
    [06] The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
    [07] This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [08] The cust

In [8]:
# ── 15-run failure analysis ───────────────────────────────────────────────────
print("Running each v1 prompt 15 times (full failure analysis) ...")

sentiment_15  = run_prompt_n_times(sentiment_prompt_v1,  n=15)
product_15    = run_prompt_n_times(product_prompt_v1,    n=15)
extraction_15 = run_prompt_n_times(extraction_prompt_v1, n=15)

analyze_consistency(sentiment_15,  "Sentiment v1 — 15 runs")
analyze_consistency(product_15,    "Product v1   — 15 runs")
analyze_consistency(extraction_15, "Extraction v1 — 15 runs")

print("""
--- Failure Analysis Summary (v1) ---
Sentiment : Output varies in wording ("Positive", "This is positive", etc.) — no fixed format.
Product   : Length, tone, bullet usage all vary run-to-run.
Extraction: Fields chosen and format (prose vs. JSON vs. bullets) are inconsistent.
""")

Running each v1 prompt 15 times (full failure analysis) ...

Consistency Analysis — Sentiment v1 — 15 runs
  Total runs      : 15
  Unique responses: 8
  Top response    : 'this customer message can be classified as **positive feedback** or **customer satisfaction**.'
  Consistency     : 33.3%

  All responses:
    [01] This customer message can be classified as positive feedback or a satisfaction message.
    [02] The customer message can be classified as **positive feedback** or **customer satisfaction**.
    [03] The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [04] This customer message is positive feedback.
    [05] This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [06] This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
    [07] The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
    [08] This customer m

---
## Part 3 — Steps 6-8: Iteration 1 — Clearer Instructions + Format Constraints (v2)

In [9]:
# ── Sentiment v2: explicit format + constraint ───────────────────────────────
sentiment_prompt_v2 = """
You are a sentiment classifier. Classify the sentiment of the customer message below.

Rules:
- Reply with exactly ONE word.
- The word must be one of: Positive, Negative, Neutral
- Do NOT add punctuation, explanation, or extra text.

Customer message: "I love this product! It's exactly what I needed."
"""

sent_v2 = run_prompt_n_times(sentiment_prompt_v2, n=15)
analyze_consistency(sent_v2, "Sentiment v2 — 15 runs")


Consistency Analysis — Sentiment v2 — 15 runs
  Total runs      : 15
  Unique responses: 1
  Top response    : 'positive'
  Consistency     : 100.0%

  All responses:
    [01] Positive
    [02] Positive
    [03] Positive
    [04] Positive
    [05] Positive
    [06] Positive
    [07] Positive
    [08] Positive
    [09] Positive
    [10] Positive
    [11] Positive
    [12] Positive
    [13] Positive
    [14] Positive
    [15] Positive


In [10]:
# ── Product v2: structured requirements ──────────────────────────────────────
product_prompt_v2 = """
Write a product description for an e-commerce listing. Follow this exact structure:

**[Product Name]**
<One compelling sentence hook — max 20 words.>

Key Features:
- <Feature 1>
- <Feature 2>
- <Feature 3>

Price: $29.99

Product: Wireless Mouse
Tone: Friendly and professional
Total length: 80-120 words
"""

prod_v2 = run_prompt_n_times(product_prompt_v2, n=15)
analyze_consistency(prod_v2, "Product v2 — 15 runs")


Consistency Analysis — Product v2 — 15 runs
  Total runs      : 15
  Unique responses: 15
  Top response    : '**wireless mouse**  \nexperience seamless navigation and comfort with our sleek and ergonomic wireless mouse, perfect for any workspace.\n\nkey features:  \n- **ergonomic design**: contoured shape provides optimal comfort for long hours of use.  \n- **reliable wireless connection**: enjoy uninterrupted performance with a powerful 2.4ghz wireless technology.  \n- **long battery life**: go for months without changing batteries, thanks to our energy-efficient design.  \n\nprice: $29.99  \n\nupgrade your setup with our wireless mouse today, combining style and functionality for an enhanced productivity experience!'
  Consistency     : 6.7%

  All responses:
    [01] **Wireless Mouse**  
Experience seamless navigation and comfort with our sleek and ergonomic wireless mouse, perfect for
    [02] **Wireless Mouse**  
Experience seamless navigation and ultimate comfort with our sleek

In [11]:
# ── Extraction v2: structured JSON output ────────────────────────────────────
extraction_prompt_v2 = """
Extract structured data from the customer feedback below.
Return ONLY a valid JSON object with these exact keys:
  order_id, order_date, delivery_rating, packaging_rating, comments

Use null for any field not mentioned. Do not add extra keys or explanation.

Customer feedback:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

extr_v2 = run_prompt_n_times(extraction_prompt_v2, n=15)
analyze_consistency(extr_v2, "Extraction v2 — 15 runs")


Consistency Analysis — Extraction v2 — 15 runs
  Total runs      : 15
  Unique responses: 7
  Top response    : '{\n  "order_id": "12345",\n  "order_date": "march 15th",\n  "delivery_rating": 5,\n  "packaging_rating": 2,\n  "comments": "the delivery was fast but the packaging was damaged."\n}'
  Consistency     : 53.3%

  All responses:
    [01] ```json
{
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": 5,
  "packaging_rating": 2,
  "comme
    [02] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": 5,
  "packaging_rating": 2,
  "comments": "T
    [03] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": 5,
  "packaging_rating": 2,
  "comments": "T
    [04] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": 5,
  "packaging_rating": 2,
  "comments": "T
    [05] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "fast",
  "packaging_rating": "damaged",
  "
    [06] {

---
## Part 4 — Steps 9-11: Iteration 2 — Few-Shot + Chain-of-Thought (v3)

In [12]:
# ── Sentiment v3: few-shot examples ─────────────────────────────────────────
sentiment_prompt_v3 = """
You are a sentiment classifier. Classify the sentiment of the customer message.
Reply with exactly ONE word: Positive, Negative, or Neutral.
No punctuation, no explanation.

Examples:
Message: "This broke after one day. Very disappointed."
Sentiment: Negative

Message: "It works fine, nothing special."
Sentiment: Neutral

Message: "Best purchase I've made this year! Absolutely love it."
Sentiment: Positive

Now classify:
Message: "I love this product! It's exactly what I needed."
Sentiment:"""

sent_v3 = run_prompt_n_times(sentiment_prompt_v3, n=15)
analyze_consistency(sent_v3, "Sentiment v3 (few-shot) — 15 runs")


Consistency Analysis — Sentiment v3 (few-shot) — 15 runs
  Total runs      : 15
  Unique responses: 1
  Top response    : 'positive'
  Consistency     : 100.0%

  All responses:
    [01] Positive
    [02] Positive
    [03] Positive
    [04] Positive
    [05] Positive
    [06] Positive
    [07] Positive
    [08] Positive
    [09] Positive
    [10] Positive
    [11] Positive
    [12] Positive
    [13] Positive
    [14] Positive
    [15] Positive


In [13]:
# ── Extraction v3: Chain-of-Thought ─────────────────────────────────────────
extraction_prompt_v3 = """
You are a data extraction specialist. Extract structured information from customer feedback.

Think step by step:
1. Read the feedback carefully.
2. Identify: order ID, order date, delivery experience, packaging condition, any other comments.
3. Map your findings to the JSON schema below.
4. Output ONLY the JSON — no other text.

Schema:
{
  "order_id": "string or null",
  "order_date": "string or null",
  "delivery_rating": "positive | negative | neutral | null",
  "packaging_rating": "positive | negative | neutral | null",
  "comments": "string or null"
}

Example:
Feedback: "Order #99887 arrived 2 days early — great packaging too!"
Output:
{
  "order_id": "99887",
  "order_date": null,
  "delivery_rating": "positive",
  "packaging_rating": "positive",
  "comments": "Arrived 2 days early."
}

Now extract:
Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
Output:"""

extr_v3 = run_prompt_n_times(extraction_prompt_v3, n=15)
analyze_consistency(extr_v3, "Extraction v3 (CoT + few-shot) — 15 runs")


Consistency Analysis — Extraction v3 (CoT + few-shot) — 15 runs
  Total runs      : 15
  Unique responses: 4
  Top response    : '{\n  "order_id": "12345",\n  "order_date": "march 15th",\n  "delivery_rating": "positive",\n  "packaging_rating": "negative",\n  "comments": "the delivery was fast but the packaging was damaged."\n}'
  Consistency     : 80.0%

  All responses:
    [01] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "positive",
  "packaging_rating": "negative"
    [02] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "positive",
  "packaging_rating": "negative"
    [03] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "positive",
  "packaging_rating": "negative"
    [04] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "positive",
  "packaging_rating": "negative"
    [05] {
  "order_id": "12345",
  "order_date": "March 15th",
  "delivery_rating": "positive",
  "packa

In [14]:
# ── Product v3: few-shot + structured output ─────────────────────────────────
product_prompt_v3 = """
You write concise, compelling e-commerce product descriptions.
Always follow the exact template shown in the examples below.

--- EXAMPLE 1 ---
Product: Bluetooth Keyboard | Price: $49.99
**Slim Bluetooth Keyboard**
Type faster and further — up to 33 ft of wireless freedom.
Key Features:
- Ultra-slim profile fits any desk setup
- 6-month battery life on a single charge
- Compatible with Windows, Mac, iOS, and Android
Price: $49.99

--- EXAMPLE 2 ---
Product: USB-C Hub | Price: $19.99
**7-in-1 USB-C Hub**
Turn one port into seven — instantly expand your laptop's capabilities.
Key Features:
- 4K HDMI output for crystal-clear displays
- 100W pass-through charging
- Plug-and-play — no drivers needed
Price: $19.99

--- NOW WRITE ---
Product: Wireless Mouse | Price: $29.99
"""

prod_v3 = run_prompt_n_times(product_prompt_v3, n=15)
analyze_consistency(prod_v3, "Product v3 (few-shot) — 15 runs")


Consistency Analysis — Product v3 (few-shot) — 15 runs
  Total runs      : 15
  Unique responses: 15
  Top response    : '**ergonomic wireless mouse**  \nnavigate seamlessly with precision and comfort — designed for all-day use.  \nkey features:  \n- adjustable dpi settings for customizable sensitivity  \n- long-lasting battery life of up to 12 months  \n- compact design perfect for travel and workspace organization  \nprice: $29.99'
  Consistency     : 6.7%

  All responses:
    [01] **Ergonomic Wireless Mouse**  
Navigate seamlessly with precision and comfort — designed for all-day use.  
Key Features
    [02] **Ergonomic Wireless Mouse**  
Experience precision and comfort for all-day productivity — no wires, no limits.  
Key Fe
    [03] **Ergonomic Wireless Mouse**  
Experience seamless navigation with precision and comfort in every click.  
Key Features:
    [04] **Ergonomic Wireless Mouse**  
Navigate with ease — enjoy seamless control and comfort all day long.  
Key Features:  


---
## Part 5 — Steps 12-13: Task Variations + Final Comparison

In [15]:
# ── Task variations — reuse v3 prompts with new inputs ───────────────────────

def make_sentiment_v3(message: str) -> str:
    return sentiment_prompt_v3.replace(
        'Message: "I love this product! It\'s exactly what I needed."',
        f'Message: "{message}"'
    )

def make_extraction_v3(feedback: str) -> str:
    return extraction_prompt_v3.replace(
        'Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."',
        f'Feedback: "{feedback}"'
    )

# Sentiment variations
variations_sentiment = [
    "The shipping took three weeks. I'm not happy at all.",
    "Package arrived on time. Does what it says on the box.",
    "Incredible quality — I've already ordered two more!",
]

for msg in variations_sentiment:
    r = run_prompt_n_times(make_sentiment_v3(msg), n=5)
    analyze_consistency(r, f"Sentiment variation: '{msg[:50]}...'")

# Extraction variations
variations_extraction = [
    "Order #77654 placed on June 3rd. The item was missing from the box!",
    "Got my order yesterday. Packaging was excellent and delivery was on time.",
]

for fb in variations_extraction:
    r = run_prompt_n_times(make_extraction_v3(fb), n=5)
    analyze_consistency(r, f"Extraction variation: '{fb[:50]}...'")


Consistency Analysis — Sentiment variation: 'The shipping took three weeks. I'm not happy at al...'
  Total runs      : 5
  Unique responses: 1
  Top response    : 'negative'
  Consistency     : 100.0%

  All responses:
    [01] Negative
    [02] Negative
    [03] Negative
    [04] Negative
    [05] Negative

Consistency Analysis — Sentiment variation: 'Package arrived on time. Does what it says on the ...'
  Total runs      : 5
  Unique responses: 1
  Top response    : 'positive'
  Consistency     : 100.0%

  All responses:
    [01] Positive
    [02] Positive
    [03] Positive
    [04] Positive
    [05] Positive

Consistency Analysis — Sentiment variation: 'Incredible quality — I've already ordered two more...'
  Total runs      : 5
  Unique responses: 1
  Top response    : 'positive'
  Consistency     : 100.0%

  All responses:
    [01] Positive
    [02] Positive
    [03] Positive
    [04] Positive
    [05] Positive

Consistency Analysis — Extraction variation: 'Order #77654 placed 

In [16]:
# ── Final comparison report ───────────────────────────────────────────────────
def consistency_pct(results: list[str]) -> float:
    normalized = [r.strip().lower() for r in results]
    counts = Counter(normalized)
    top_count = counts.most_common(1)[0][1]
    return round((top_count / len(results)) * 100, 1)

print("\n" + "="*65)
print("  FINAL COMPARISON REPORT (15-run consistency %)")
print("="*65)
print(f"{'Task':<25} {'v1 (zero-shot)':>14} {'v2 (+format)':>13} {'v3 (+few-shot/CoT)':>19}")
print("-"*65)

tasks = [
    ("Sentiment Analysis",       sentiment_15,  sent_v2,  sent_v3),
    ("Product Description",      product_15,    prod_v2,  prod_v3),
    ("Data Extraction",          extraction_15, extr_v2,  extr_v3),
]

for name, v1, v2, v3 in tasks:
    print(f"{name:<25} {consistency_pct(v1):>13}%  {consistency_pct(v2):>12}%  {consistency_pct(v3):>18}%")

print("="*65)
print("""
Key takeaways:
  1. Zero-shot prompts (v1) produce highly variable outputs.
  2. Adding explicit format constraints (v2) significantly raises consistency.
  3. Few-shot examples and Chain-of-Thought (v3) push consistency above 80%%
     for all three task types.
  4. For classification tasks, a few clear examples are the most impactful lever.
  5. For extraction tasks, CoT + a schema example locks in both format and logic.
""")


  FINAL COMPARISON REPORT (15-run consistency %)
Task                      v1 (zero-shot)  v2 (+format)  v3 (+few-shot/CoT)
-----------------------------------------------------------------
Sentiment Analysis                 33.3%         100.0%               100.0%
Product Description                 6.7%           6.7%                 6.7%
Data Extraction                    26.7%          53.3%                80.0%

Key takeaways:
  1. Zero-shot prompts (v1) produce highly variable outputs.
  2. Adding explicit format constraints (v2) significantly raises consistency.
  3. Few-shot examples and Chain-of-Thought (v3) push consistency above 80%%
     for all three task types.
  4. For classification tasks, a few clear examples are the most impactful lever.
  5. For extraction tasks, CoT + a schema example locks in both format and logic.

